# K-NN no Titanic — FATEC

Atividade da aula de 18-08-2026. Neste roteiro preparamos o `train.csv` original da competição Titanic no Kaggle e treinamos k-Nearest Neighbors (k-NN) para prever `Survived`.

## Como o k-NN decide

Cada passageiro é representado por um vetor de atributos, por exemplo idade, tarifa e sexo. Para classificar um novo vetor, o k-NN calcula sua distância (por padrão, Euclidiana) até os vetores de treino, encontra os **k** vizinhos mais próximos e usa a votação majoritária de `Survived`. Por depender diretamente das distâncias, o método exige números, ausência de valores ausentes e escalas comparáveis.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.neighbors import KNeighborsClassifier

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'data' / 'train.csv').exists() and (PROJECT_ROOT.parent / 'data' / 'train.csv').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.preprocessing import (
    REQUIRED_COLUMNS, apply_min_max, fit_min_max, fit_preprocessor,
    transform_features, validate_titanic_dataset,
)

DATA_PATH = PROJECT_ROOT / 'data' / 'train.csv'
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f'Dataset obrigatório não encontrado em {DATA_PATH}. Coloque aqui o train.csv original do Titanic — Kaggle; nenhuma fonte alternativa é usada.'
    )

print(f'Raiz do projeto: {PROJECT_ROOT}')
print(f'Dataset: {DATA_PATH}')

## Exploração inicial e validação do CSV

Antes de modelar, confirmamos as colunas esperadas e observamos dimensões, tipos, exemplos, estatísticas, valores nulos e categorias. Isso evita preparar acidentalmente outro arquivo parecido.

In [ ]:
titanic = pd.read_csv(DATA_PATH)
validate_titanic_dataset(titanic)

print(f'Linhas: {titanic.shape[0]} | Colunas: {titanic.shape[1]}')
print('\nTipos das colunas:')
display(titanic.dtypes.to_frame('dtype'))
print('\nPrimeiras linhas:')
display(titanic.head())
print('\nEstatísticas básicas:')
display(titanic.describe(include='all').T)
print('\nValores nulos por coluna:')
display(titanic.isna().sum().to_frame('nulos'))
print('\nCategorias relevantes:')
for column in ['Survived', 'Pclass', 'Sex', 'Embarked']:
    print(f'{column}: {titanic[column].value_counts(dropna=False).to_dict()}')

assert REQUIRED_COLUMNS.issubset(titanic.columns)

## Seleção de atributos

`PassengerId` é apenas identificador; `Name`, `Ticket` e `Cabin` são strings sem uma distância matemática direta útil neste exercício. Eles não entrarão no modelo. `Survived` não é removido: é o alvo que queremos prever.

In [ ]:
target = titanic['Survived'].copy()
raw_features = titanic.drop(columns=['Survived', 'PassengerId', 'Name', 'Ticket', 'Cabin'])
print('Colunas candidatas ao modelo:', raw_features.columns.tolist())
assert 'Survived' not in raw_features.columns

## Separação treino/teste antes das transformações

`X` contém os atributos e `y` contém `Survived`. Reservamos o teste para a avaliação final e usamos `stratify=y` para preservar a proporção entre sobreviventes e não sobreviventes. A mediana, moda e parâmetros de escala são ajustados apenas no treino: usar também o teste seria *data leakage*.

In [ ]:
train_raw, test_raw, y_train, y_test = train_test_split(
    titanic, target, test_size=0.20, random_state=42, stratify=target
)
print(f'Treino: {train_raw.shape[0]} linhas | Teste: {test_raw.shape[0]} linhas')
print('Proporção Survived (treino/teste):', round(y_train.mean(), 3), round(y_test.mean(), 3))

## Valores ausentes, encoding e engenharia de atributos

O k-NN não aceita `NaN`. Preenchemos `Age` pela mediana do treino e `Embarked` pela moda do treino. `Sex` é convertido por Pandas em `male → 0` e `female → 1`. `Embarked` e `Pclass` recebem dummies com `pd.get_dummies`; embora Pclass use 1, 2 e 3, aqui são categorias, não uma distância contínua. Para `Embarked`, C é a base e Q/S são indicadores. Criamos também `FamilySize = SibSp + Parch + 1`, usando-a no modelo expandido no lugar de SibSp e Parch.

In [ ]:
preprocessor = fit_preprocessor(train_raw)
train_prepared = transform_features(train_raw, preprocessor, use_family_size=True)
test_prepared = transform_features(test_raw, preprocessor, use_family_size=True)

print('Mediana de Age ajustada no treino:', preprocessor.age_median)
print('Moda de Embarked ajustada no treino:', preprocessor.embarked_mode)
display(train_prepared.head())
print('NaN no treino preparado:', int(train_prepared.isna().sum().sum()))
print('NaN no teste preparado:', int(test_prepared.isna().sum().sum()))
print('Tipos finais:', train_prepared.dtypes.to_dict())
assert train_prepared.isna().sum().sum() == 0
assert test_prepared.isna().sum().sum() == 0
assert all(pd.api.types.is_numeric_dtype(dtype) for dtype in train_prepared.dtypes)

## Min-Max Scaling manual

A distância Euclidiana soma diferenças ao quadrado. `Pclass` tem escala curta (1–3), enquanto `Fare` pode ser muito maior; sem escala, Fare poderia dominar a distância. A fórmula manual exigida é `x_norm = (x - x_min) / (x_max - x_min)`. As dummies e `Sex` já são 0/1; normalizamos as contínuas `Age`, `Fare` e, no modelo C, `FamilySize`. Os mínimos/máximos são obtidos só no treino e reutilizados no teste.

In [ ]:
display(train_raw[['Pclass', 'Fare']].describe().T)

# Exemplo explícito da fórmula manual em Pandas (ajustada somente no treino).
scaling_example = fit_min_max(train_prepared, ['Age', 'Fare', 'FamilySize'])
display(apply_min_max(train_prepared[['Age', 'Fare', 'FamilySize']], scaling_example).head())

## Escolha de k com validação cruzada

`k` é o número de vizinhos que votam. Com `k=1`, um ponto ruidoso pode decidir a classe; um k muito grande suaviza demais as fronteiras. Valores ímpares reduzem empates em classificação binária. Avaliamos 1, 3, 5, 7, 9, 11, 13 e 15 somente dentro do treino com validação cruzada estratificada e escolhemos o melhor; o teste permanece intocado até a avaliação final.

In [ ]:
K_VALUES = [1, 3, 5, 7, 9, 11, 13, 15]
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def build_experiment(features, continuous_columns):
    """Seleciona colunas e aplica Min-Max com parâmetros exclusivos do treino."""
    X_train = train_prepared.loc[:, features].copy()
    X_test = test_prepared.loc[:, features].copy()
    min_max = fit_min_max(X_train, continuous_columns)
    return apply_min_max(X_train, min_max), apply_min_max(X_test, min_max)

def evaluate_knn(name, features, continuous_columns):
    X_train_exp, X_test_exp = build_experiment(features, continuous_columns)
    cv_scores = {
        k: cross_val_score(KNeighborsClassifier(n_neighbors=k), X_train_exp, y_train, cv=cv, scoring='accuracy').mean()
        for k in K_VALUES
    }
    best_k = max(cv_scores, key=lambda k: (cv_scores[k], -k))
    model = KNeighborsClassifier(n_neighbors=best_k)
    model.fit(X_train_exp, y_train)
    prediction = model.predict(X_test_exp)
    return {
        'name': name, 'features': features, 'k': best_k, 'cv_scores': cv_scores,
        'accuracy': accuracy_score(y_test, prediction),
        'confusion_matrix': confusion_matrix(y_test, prediction),
        'report': classification_report(y_test, prediction, digits=3),
    }

## Experimentos A, B e C

A é o baseline com idade e tarifa. B acrescenta sexo. C é o modelo expandido com sexo, as dummies de classe/embarque e tamanho da família. As métricas abaixo são avaliações finais no conjunto de teste reservado.

In [ ]:
pclass_dummies = [column for column in train_prepared.columns if column.startswith('Pclass_')]
embarked_dummies = [column for column in train_prepared.columns if column.startswith('Embarked_')]

experiments = {
    'A': evaluate_knn('Experimento A', ['Age', 'Fare'], ['Age', 'Fare']),
    'B': evaluate_knn('Experimento B', ['Age', 'Fare', 'Sex'], ['Age', 'Fare']),
    'C': evaluate_knn(
        'Experimento C',
        ['Age', 'Fare', 'Sex', 'FamilySize', *pclass_dummies, *embarked_dummies],
        ['Age', 'Fare', 'FamilySize'],
    ),
}

for label, result in experiments.items():
    print(f"{result['name']} | features={result['features']} | k={result['k']} | accuracy={result['accuracy']:.4f}")
    print('Matriz de confusão (linhas: real 0/1; colunas: previsto 0/1):')
    print(result['confusion_matrix'])
    print(result['report'])

## Gráficos e comparação

O primeiro gráfico mostra o desempenho médio da validação cruzada (somente treino) para cada k. O segundo compara a acurácia final dos três experimentos. A conclusão deve seguir os números executados, sem forçar a hipótese de que `Sex` melhora necessariamente o resultado.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for label, result in experiments.items():
    axes[0].plot(K_VALUES, [result['cv_scores'][k] for k in K_VALUES], marker='o', label=f"{label} (k={result['k']})")
axes[0].set_title('Validação cruzada: acurácia por k')
axes[0].set_xlabel('Número de vizinhos (k)')
axes[0].set_ylabel('Acurácia média no treino')
axes[0].set_xticks(K_VALUES)
axes[0].legend()
axes[0].grid(alpha=0.3)

labels = list(experiments)
accuracies = [experiments[label]['accuracy'] for label in labels]
bars = axes[1].bar(labels, accuracies, color=['#4C78A8', '#F58518', '#54A24B'])
axes[1].set_title('Acurácia final por experimento')
axes[1].set_xlabel('Experimento')
axes[1].set_ylabel('Acurácia no teste')
axes[1].set_ylim(0, 1)
for bar, value in zip(bars, accuracies):
    axes[1].text(bar.get_x() + bar.get_width() / 2, value + 0.02, f'{value:.3f}', ha='center')

plt.tight_layout()
plt.show()

## Resultado consolidado e verificações finais

A tabela registra atributos, k escolhido por validação, acurácia e diferença em relação ao baseline A. As verificações garantem que `Survived` não entrou em `X`, que não há NaN/string no k-NN e que a normalização foi ajustada no treino.

In [ ]:
summary = pd.DataFrame([
    {
        'Experimento': label,
        'Features': ', '.join(result['features']),
        'k escolhido': result['k'],
        'Acurácia teste': result['accuracy'],
        'Diferença vs A': result['accuracy'] - experiments['A']['accuracy'],
    }
    for label, result in experiments.items()
])
display(summary)

assert 'Survived' not in train_prepared.columns
assert train_prepared.isna().sum().sum() == 0
assert test_prepared.isna().sum().sum() == 0
assert all(pd.api.types.is_numeric_dtype(dtype) for dtype in train_prepared.dtypes)
print('Validações finais: PASS — sem NaN, apenas números, alvo fora de X e escala ajustada no treino.')